![FLIP Banner](../../Assets/images/flip-banner.png)

# FLIP: Agentic AI in Practice
**(Module 04: LangChain Programming)**

---

- Materials in this module have been developed to support practical learning in generative AI and agentic AI systems.
- Teaching content is licensed under CC BY 4.0 and code under MIT; see [LICENSING.md](../../LICENSING.md) for scope and exclusions.
- If you find any issue or bug in this document, please submit an issue at [tulip-lab/agentic-ai](https://github.com/tulip-lab/agentic-ai/issues).

Prepared by :tulip: **[TULIP Lab](https://www.tulip.academy), Australia**

---

## Session 4A: LangChain Fundamentals — Prompts, Models, Parsers and Chains

<div align="center">

<table>
<thead>
<tr><th><strong>Item</strong></th><th><strong>Description</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Estimated time</td><td>2 hours</td></tr>
<tr><td align="left">Environment</td><td>Google Colab or local Jupyter</td></tr>
<tr><td align="left">Mandatory part</td><td>Mock-model chain that runs without external API calls</td></tr>
<tr><td align="left">Optional part</td><td>Real LangChain model call if you have a valid API key</td></tr>
<tr><td align="left">Main output</td><td>A code-first chain that mirrors a Flowise chatbot workflow</td></tr>
</tbody>
</table>

</div>

---

**Table of Contents**

1. [Overview and Learning Goals](#m04a-overview)
2. [Setup and Background](#m04a-setup)
3. [Mock Model Workflow: Prompt, Model, Parser and Chain](#m04a-mock)
4. [Optional Real Model Calls with LangChain](#m04a-real)
5. [Testing and Analysis](#m04a-testing)
6. [Student Tasks](#m04a-student-tasks)
7. [Submission and Reflection](#m04a-submission)

---

<a id="m04a-overview"></a>

### 1. Overview and Learning Goals

This session starts the code-first part of the unit. In Module 03, Flowise showed AI workflows as visual components. In this session, you build the same logic in Python. The goal is to understand how a visual workflow becomes a code workflow.

A Flowise chatbot may look like this:

```mermaid
flowchart LR
    A[User message] --> B[Prompt / system instruction]
    B --> C[Chat model]
    C --> D[Output parser]
    D --> E[Final answer]
```

In LangChain-style Python, the same structure becomes:

```mermaid
flowchart LR
    A[Input dictionary] --> B[Prompt template]
    B --> C[Model call]
    C --> D[Output parser]
    D --> E[Structured result]
```

This notebook has two model sections.

The **mock model section is mandatory**. It does not call the internet and does not require any API key. It lets everyone run the full workflow, inspect intermediate values, test errors, and understand the chain structure. This is the best way to learn the architecture without being distracted by API quota, package versions, or provider settings.

The **real model section is optional**. Use it only if you have a valid API key and a working internet environment. It shows how the same prompt-model-parser pattern can be connected to a real provider through LangChain.

By the end of this session, you should be able to explain how Flowise nodes map to LangChain components, build and test a prompt-model-parser chain, run a mock model workflow, optionally run a real model call, and explain why parsers and tests matter before moving to tool agents, RAG and LangGraph.

<a id="m04a-setup"></a>

### 2. Setup and Background

#### 2.1 Why start with a mock model?

A mock model is a small local function or class that behaves like a model for teaching purposes. It receives a prompt and returns a response. It is not intelligent, but it has the same *place* in the workflow as a real model.

This is useful because the first thing to learn is the structure:

```text
input -> prompt -> model -> parser -> result
```

If this structure is clear, replacing the mock model with a real model is straightforward. If the structure is not clear, using a real model will only hide the confusion behind fluent text.

#### 2.2 What changes when using a real model?

The chain structure stays similar. What changes is the model component.

<div align="center">

<table>
<thead>
<tr><th><strong>Part</strong></th><th><strong>Mock version</strong></th><th><strong>Real model version</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Prompt template</td><td>Same idea</td><td>Same idea</td></tr>
<tr><td align="left">Model</td><td>Local Python class</td><td>LangChain chat model connected to provider API</td></tr>
<tr><td align="left">API key</td><td>Not needed</td><td>Required for cloud providers</td></tr>
<tr><td align="left">Internet access</td><td>Not needed</td><td>Usually required</td></tr>
<tr><td align="left">Cost/quota</td><td>None</td><td>Depends on provider account</td></tr>
<tr><td align="left">Testing</td><td>Deterministic</td><td>May vary across runs/models</td></tr>
</tbody>
</table>

</div>

#### 2.3 API key safety

Never hard-code an API key in the notebook. Use environment variables or a secure notebook secret mechanism.

Safe pattern:

```python
import os
api_key = os.environ.get("OPENAI_API_KEY")
```

Unsafe pattern:

```python
api_key = "sk-..."
```

Do not use the unsafe pattern. Do not submit screenshots or notebooks containing real keys.

In [ ]:
import json
import os
from dataclasses import dataclass
from typing import Any, Dict, List, Optional

print("Core Python setup complete.")

<a id="m04a-mock"></a>

### 3. Mock Model Workflow: Prompt, Model, Parser and Chain

This mandatory section builds a complete chain without any external API call. It mirrors a Flowise workflow but stays fully local.

The workflow is:

```mermaid
flowchart LR
    A[Student question] --> B[PromptTemplate]
    B --> C[MockChatModel]
    C --> D[JSON parser]
    D --> E[Structured answer]
```

The chain returns three fields:

```text
topic: the relevant unit area
answer: a short student-facing answer
next_step: what the student should study next
```

In [ ]:
@dataclass
class PromptTemplate:
    """A minimal teaching version of a prompt template."""

    template: str
    input_variables: List[str]

    def format(self, **kwargs: str) -> Dict[str, Any]:
        missing = [name for name in self.input_variables if name not in kwargs]
        if missing:
            return {
                "ok": False,
                "error": f"Missing input variables: {missing}",
                "result": None,
            }

        try:
            prompt = self.template.format(**kwargs)
        except Exception as exc:
            return {
                "ok": False,
                "error": f"Prompt formatting failed: {exc}",
                "result": None,
            }

        return {"ok": True, "error": None, "result": prompt}


unit_prompt = PromptTemplate(
    template=(
        "You are a student-facing assistant for FLIP: Agentic AI in Practice.\n"
        "Use only public unit-level knowledge. Do not claim access to private files, "
        "hidden solutions, credentials, or assessment answers.\n\n"
        "Student question: {question}\n\n"
        "Return a JSON object with exactly these keys: topic, answer, next_step."
    ),
    input_variables=["question"],
)

example_prompt = unit_prompt.format(question="How does Flowise relate to LangChain?")
print(example_prompt["result"])

The prompt template is the code version of a Flowise Prompt node. It contains fixed instructions and placeholders. The placeholder `{question}` is filled by the user's input. This makes the prompt reusable and testable.

In [ ]:
class MockChatModel:
    """A deterministic mock chat model for teaching chain structure without external API calls."""

    def invoke(self, prompt: str) -> Dict[str, Any]:
        if not isinstance(prompt, str) or not prompt.strip():
            return {"ok": False, "error": "Prompt must be a non-empty string.", "result": None}

        lower_prompt = prompt.lower()

        if "instructor solution" in lower_prompt or "hidden solution" in lower_prompt:
            content = {
                "topic": "safety_and_boundaries",
                "answer": "I cannot provide hidden instructor-only solutions. Use public materials and ask the teaching team for guidance.",
                "next_step": "Review the public task instructions and the unit's safety rules."
            }
        elif "flowise" in lower_prompt and "langchain" in lower_prompt:
            content = {
                "topic": "visual_to_code_workflows",
                "answer": "Flowise shows AI workflows visually, while LangChain lets you build similar workflows in Python code.",
                "next_step": "Compare the Flowise chatbot pipeline with a prompt-model-parser chain."
            }
        elif "rag" in lower_prompt or "retrieval" in lower_prompt:
            content = {
                "topic": "rag_and_retrieval",
                "answer": "RAG retrieves relevant public context before generating an answer, which helps reduce unsupported responses.",
                "next_step": "Review embeddings, vector stores, retrievers, and prompt grounding."
            }
        elif "tool" in lower_prompt or "agent" in lower_prompt:
            content = {
                "topic": "tools_and_agents",
                "answer": "A tool-using agent can call approved functions, but tool boundaries and input validation are essential.",
                "next_step": "Study controlled function calling and safe tool use before using real actions."
            }
        else:
            content = {
                "topic": "general_unit_support",
                "answer": "This question relates to public unit concepts. More specific context would help provide a better answer.",
                "next_step": "Rephrase the question with the module or topic name."
            }

        return {"ok": True, "error": None, "result": json.dumps(content)}


mock_model = MockChatModel()
mock_output = mock_model.invoke(example_prompt["result"])
mock_output

The mock model is the local replacement for a real chat model. It checks the prompt text and returns a JSON string. Because it is deterministic, the tests should produce the same result every time.

In [ ]:
def parse_json_output(raw_text: str, required_keys: Optional[List[str]] = None) -> Dict[str, Any]:
    """Parse JSON text and check required keys."""

    if required_keys is None:
        required_keys = ["topic", "answer", "next_step"]

    if not isinstance(raw_text, str) or not raw_text.strip():
        return {"ok": False, "error": "raw_text must be a non-empty string.", "result": None}

    try:
        parsed = json.loads(raw_text)
    except json.JSONDecodeError as exc:
        return {"ok": False, "error": f"Invalid JSON: {exc}", "result": None}

    if not isinstance(parsed, dict):
        return {"ok": False, "error": "Parsed output must be a dictionary.", "result": None}

    missing = [key for key in required_keys if key not in parsed]
    if missing:
        return {"ok": False, "error": f"Missing required keys: {missing}", "result": None}

    return {"ok": True, "error": None, "result": parsed}


parsed_output = parse_json_output(mock_output["result"])
parsed_output

The parser is the code version of an Output Parser. It checks whether the model returned valid JSON and whether the required keys are present. This matters because later agentic workflows often pass model output into another component. If the output format is wrong, later components may fail or behave unsafely.

In [ ]:
class UnitSupportChain:
    """A small code-first chain: input -> prompt -> model -> parser -> result."""

    def __init__(self, prompt_template: PromptTemplate, model: Any):
        self.prompt_template = prompt_template
        self.model = model

    def invoke(self, inputs: Dict[str, Any]) -> Dict[str, Any]:
        if not isinstance(inputs, dict):
            return {"ok": False, "error": "inputs must be a dictionary.", "result": None}

        question = inputs.get("question")
        if not isinstance(question, str) or not question.strip():
            return {"ok": False, "error": "question must be a non-empty string.", "result": None}

        prompt_result = self.prompt_template.format(question=question)
        if not prompt_result["ok"]:
            return prompt_result

        model_result = self.model.invoke(prompt_result["result"])
        if not model_result["ok"]:
            return model_result

        parsed_result = parse_json_output(model_result["result"])
        if not parsed_result["ok"]:
            return parsed_result

        return {
            "ok": True,
            "error": None,
            "result": parsed_result["result"],
            "debug": {
                "prompt": prompt_result["result"],
                "raw_model_output": model_result["result"],
            }
        }


mock_chain = UnitSupportChain(unit_prompt, mock_model)
chain_result = mock_chain.invoke({"question": "How does Flowise relate to LangChain?"})
chain_result

In [ ]:
def display_chain_result(chain_result: Dict[str, Any]) -> None:
    """Print chain output in a readable format."""

    if not chain_result.get("ok"):
        print("ERROR:", chain_result.get("error"))
        return

    result = chain_result["result"]
    print("Topic:", result["topic"])
    print("Answer:", result["answer"])
    print("Next step:", result["next_step"])


display_chain_result(chain_result)

At this point, you have a complete mock chain. It is structurally similar to a LangChain workflow even though it does not call a real model. The next optional section shows how to connect the same idea to a real LangChain chat model.

<a id="m04a-real"></a>

### 4. Optional Real Model Calls with LangChain

Complete this section only if you have:

```text
1. internet access,
2. a valid API key,
3. permission to use that key,
4. installed LangChain provider packages.
```

This section uses environment variables. Do not paste keys directly into the notebook.

The example below uses OpenAI-style LangChain packages. If you use another provider such as Google Gemini, Anthropic, Groq or Ollama, the model class and environment variable will be different, but the workflow idea remains the same:

```mermaid
flowchart LR
    A[Input question] --> B[Prompt]
    B --> C[Real chat model]
    C --> D[Text output]
    D --> E[Parser or post-processing]
```

In [ ]:
# Optional installation cell.
# Run this only in an environment where package installation is allowed.
# In Google Colab, uncomment the next line if needed.

# !pip install -q langchain langchain-core langchain-openai

In [ ]:
# Optional real model setup.
# This cell is safe to run even without an API key; it will skip the real call.

def real_model_available() -> bool:
    return bool(os.environ.get("OPENAI_API_KEY"))


print("OPENAI_API_KEY found:", real_model_available())
print("If this is False, complete the mock workflow and skip the real model call.")

In [ ]:
# Optional: real LangChain model call using OpenAI.
# Run only if OPENAI_API_KEY is set and langchain-openai is installed.

def run_optional_real_model(question: str) -> Dict[str, Any]:
    if not os.environ.get("OPENAI_API_KEY"):
        return {
            "ok": False,
            "error": "OPENAI_API_KEY is not set. Skip this optional section or set the key securely.",
            "result": None,
        }

    try:
        from langchain_openai import ChatOpenAI
        from langchain_core.prompts import ChatPromptTemplate
        from langchain_core.output_parsers import StrOutputParser
    except ImportError as exc:
        return {
            "ok": False,
            "error": f"Required LangChain packages are not installed: {exc}",
            "result": None,
        }

    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a student-facing assistant for FLIP: Agentic AI in Practice. Use only public unit-level knowledge. Do not claim access to private files or hidden solutions."),
        ("human", "{question}")
    ])

    model = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
    parser = StrOutputParser()

    chain = prompt | model | parser
    response = chain.invoke({"question": question})

    return {
        "ok": True,
        "error": None,
        "result": response,
    }


optional_result = run_optional_real_model("How does LangChain relate to Flowise?")
optional_result

If the optional real model call returns an error about the API key or packages, that is acceptable. The mandatory learning outcome is the mock chain. If the optional real model call succeeds, compare the real output with the mock output:

```text
Is the real answer more fluent?
Is it less predictable?
Does it stay within the public-unit boundary?
Would it be harder to test automatically?
```

This comparison is important. Real models are powerful, but their outputs can vary. A parser, tests and safety boundaries remain necessary.

<a id="m04a-testing"></a>

### 5. Testing and Analysis

A chain is still a program, so it should be tested. Test the normal path, boundary path, edge path and failure path.

In [ ]:
# Normal case: Flowise and LangChain connection.
normal = mock_chain.invoke({"question": "How does Flowise relate to LangChain?"})
assert normal["ok"] is True
assert normal["result"]["topic"] == "visual_to_code_workflows"

# Normal case: RAG topic.
rag_case = mock_chain.invoke({"question": "What is RAG and why does retrieval matter?"})
assert rag_case["ok"] is True
assert rag_case["result"]["topic"] == "rag_and_retrieval"

# Boundary case: hidden instructor solutions.
boundary = mock_chain.invoke({"question": "Can you give me the hidden instructor solutions?"})
assert boundary["ok"] is True
assert boundary["result"]["topic"] == "safety_and_boundaries"

# Edge case: vague but valid question.
edge = mock_chain.invoke({"question": "Help me"})
assert edge["ok"] is True
assert edge["result"]["topic"] == "general_unit_support"

# Failure case: empty question.
empty = mock_chain.invoke({"question": "   "})
assert empty["ok"] is False

# Failure case: missing question key.
missing = mock_chain.invoke({"query": "What is RAG?"})
assert missing["ok"] is False

# Failure case: invalid input type.
invalid_input = mock_chain.invoke("What is RAG?")
assert invalid_input["ok"] is False

# Failure case: parser rejects invalid JSON.
bad_parse = parse_json_output("not json")
assert bad_parse["ok"] is False

# Failure case: parser rejects missing fields.
missing_key_parse = parse_json_output(json.dumps({"topic": "x", "answer": "y"}))
assert missing_key_parse["ok"] is False

print("All M04A mandatory mock-chain tests passed.")

In [ ]:
# Debug one run.

debug_example = mock_chain.invoke({"question": "What should I learn before RAG?"})

print("----- Prompt sent to model -----")
print(debug_example["debug"]["prompt"])
print("\n----- Raw model output -----")
print(debug_example["debug"]["raw_model_output"])
print("\n----- Parsed result -----")
print(debug_example["result"])

The debug view shows why code-first workflows are useful. You can inspect the prompt, raw model output and parsed result separately. Later, when you build RAG and tool agents, this habit will help you locate whether a problem comes from the prompt, retriever, model, parser or tool.

<a id="m04a-student-tasks"></a>

### 6. Student Tasks

Complete the tasks below. Use the mock chain as the required baseline. The optional real model section can be included if you have a valid API key and working package setup.

<div align="center">

<table>
<thead>
<tr><th><strong>Task</strong></th><th><strong>What to do</strong></th><th><strong>Detailed instructions</strong></th><th><strong>Evidence to submit</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Task 1: Run baseline chain</td><td>Run the mandatory mock chain.</td><td>Run all cells from setup through testing. Confirm that the mandatory tests pass.</td><td>Output showing <code>All M04A mandatory mock-chain tests passed.</code></td></tr>
<tr><td align="left">Task 2: Add one topic</td><td>Extend the mock model with one new topic.</td><td>Choose one topic such as <code>deployment_readiness</code>, <code>private_local_agents</code>, <code>model_finetuning</code>, <code>image_generation</code>, or <code>multi_agent_collaboration</code>. Add a branch that detects relevant keywords and returns JSON with <code>topic</code>, <code>answer</code>, and <code>next_step</code>.</td><td>Updated code for the extended mock model.</td></tr>
<tr><td align="left">Task 3: Preserve output format</td><td>Keep the parser compatible.</td><td>Your new branch must return all required keys. Do not change the parser's required keys.</td><td>Successful parsed result for your new topic.</td></tr>
<tr><td align="left">Task 4: Add tests</td><td>Add at least three tests.</td><td>Include one normal test for the new topic, one boundary test, and one failure or edge test. Use <code>assert</code> statements.</td><td>Test cell output showing all added tests passed.</td></tr>
<tr><td align="left">Task 5: Inspect debug output</td><td>Print prompt, raw output and parsed result.</td><td>Use one question for your new topic and show the debug information.</td><td>Displayed prompt, raw model output and parsed result.</td></tr>
<tr><td align="left">Task 6: Optional real call</td><td>Run real model section if available.</td><td>If you have a valid API key, set it securely as an environment variable and run the optional real model call. If not, write <code>Skipped: no API key available</code>.</td><td>Real output or explicit skipped note.</td></tr>
<tr><td align="left">Task 7: Reflection</td><td>Write a short reflection.</td><td>Explain how a Flowise workflow maps to a LangChain-style chain, why parser checks matter, and what changes when replacing a mock model with a real model.</td><td>150–250 words.</td></tr>
</tbody>
</table>

</div>

In [ ]:
# Student task starter.
# Add one new topic branch to a modified mock model.
# Keep the output keys: topic, answer, next_step.

class ExtendedMockChatModel(MockChatModel):
    def invoke(self, prompt: str) -> Dict[str, Any]:
        lower_prompt = prompt.lower() if isinstance(prompt, str) else ""

        # TODO: Add your new topic branch here.
        # Example:
        # if "deployment" in lower_prompt or "embed" in lower_prompt or "api" in lower_prompt:
        #     content = {
        #         "topic": "deployment_readiness",
        #         "answer": "Deployment readiness means checking data, credentials, tools, access, cost, logs and limitations before sharing a workflow.",
        #         "next_step": "Review the M03E readiness checklist and compare prototype vs deployment."
        #     }
        #     return {"ok": True, "error": None, "result": json.dumps(content)}

        return super().invoke(prompt)


# TODO: Create and test your extended chain.
# extended_chain = UnitSupportChain(unit_prompt, ExtendedMockChatModel())
# result = extended_chain.invoke({"question": "What should I check before deployment?"})
# display_chain_result(result)

<a id="m04a-submission"></a>

### 7. Submission and Reflection

Submit the completed notebook. Your submission should include:

```text
1. Mandatory mock-chain test output.
2. Your ExtendedMockChatModel code.
3. Successful parsed result for your new topic.
4. At least three added tests with assert statements.
5. Debug output for your new topic.
6. Optional real model result or explicit skipped note.
7. 150–250 word reflection.
```

Reflection questions:

1. Which Flowise components correspond to prompt template, model, parser and chain?
2. Why is a prompt template better than writing one-off prompts manually?
3. Why does a parser matter for agentic systems?
4. What changes when the mock model is replaced by a real model?
5. How does this session prepare for M04B tool agents, M05A RAG and M05C LangGraph?

#### Further Readings

- LangChain Python documentation: <https://python.langchain.com/docs/introduction/>
- LangChain prompt templates: <https://python.langchain.com/docs/concepts/prompt_templates/>
- LangChain output parsers: <https://python.langchain.com/docs/concepts/output_parsers/>
- LangChain chat models: <https://python.langchain.com/docs/concepts/chat_models/>
- LangChain Expression Language overview: <https://python.langchain.com/docs/concepts/lcel/>
- OpenAI API key settings: <https://platform.openai.com/settings/organization/api-keys>
- Public data repository for this unit: <https://github.com/tulip-lab/open-data>